# 🔍 Fake Product Review Detection — Plotly Dash Dashboard

> **NLP · Machine Learning · Sentiment Analysis**
>
> Run each cell in order, then open **http://127.0.0.1:8050** in your browser.
>
> **Requires:** `pip install dash plotly pandas joblib scikit-learn nltk`

## 1 · Imports

In [1]:
import os, string
import numpy as np
import pandas as pd
import joblib
import plotly.graph_objects as go
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State, dash_table
import warnings
warnings.filterwarnings('ignore')

c:\Users\Tarun\anaconda3\envs\mypython_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## 2 · Load Model Assets

Loads the `.pkl` pipeline files produced by the **Part-3** training notebook.
Make sure the `models/` directory exists and contains the saved pipelines.

In [2]:
def text_process(review):
    """Strip residual punctuation and tokenise (text already lemmatised in Part 1)."""
    nopunc = [char for char in str(review) if char not in string.punctuation]
    return ''.join(nopunc).split()


MODEL_DIR = os.path.join(os.getcwd(), 'Models')

MODEL_FILES = {
    'Logistic Regression':    'logistic_regression_pipeline.pkl',
    'Support Vector Machine': 'support_vector_machine_pipeline.pkl',
    'Random Forest':          'random_forest_pipeline.pkl',
    'Decision Tree':          'decision_tree_pipeline.pkl',
    'Naive Bayes':            'naive_bayes_pipeline.pkl',
    'KNN (k=5)':             'knn_k5_pipeline.pkl',
}

pipelines = {}
for name, fname in MODEL_FILES.items():
    path = os.path.join(MODEL_DIR, fname)
    if os.path.exists(path):
        pipelines[name] = joblib.load(path)
        print(f'  ✅  Loaded: {name}')

# Fallback: try best_model_pipeline.pkl
if not pipelines:
    best_path = os.path.join(MODEL_DIR, 'best_model_pipeline.pkl')
    if os.path.exists(best_path):
        pipelines['Best Model'] = joblib.load(best_path)
        print('  ✅  Loaded: Best Model (fallback)')

if not pipelines:
    raise FileNotFoundError(
        'No .pkl files found in models/.\n'
        'Run the Part-3 notebook first to generate saved pipelines.'
    )

print(f'\nTotal pipelines loaded: {len(pipelines)}')

  ✅  Loaded: Logistic Regression
  ✅  Loaded: Support Vector Machine
  ✅  Loaded: Random Forest
  ✅  Loaded: Decision Tree
  ✅  Loaded: Naive Bayes
  ✅  Loaded: KNN (k=5)

Total pipelines loaded: 6


In [3]:
# Pre-computed metrics from Part 3 — update with your actual results
METRICS = pd.DataFrame([
    {'Model': 'Logistic Regression',    'Accuracy': 88.3, 'Precision': 87.1, 'Recall': 89.6, 'F1-Score': 88.3},
    {'Model': 'Support Vector Machine', 'Accuracy': 87.9, 'Precision': 86.8, 'Recall': 89.0, 'F1-Score': 87.9},
    {'Model': 'Random Forest',          'Accuracy': 85.4, 'Precision': 84.2, 'Recall': 86.7, 'F1-Score': 85.4},
    {'Model': 'Naive Bayes',            'Accuracy': 83.1, 'Precision': 82.0, 'Recall': 84.3, 'F1-Score': 83.1},
    {'Model': 'Decision Tree',          'Accuracy': 78.2, 'Precision': 77.5, 'Recall': 79.0, 'F1-Score': 78.2},
    {'Model': 'KNN (k=5)',             'Accuracy': 73.6, 'Precision': 72.1, 'Recall': 75.2, 'F1-Score': 73.6},
]).sort_values('F1-Score', ascending=False).reset_index(drop=True)

METRIC_COLORS = {
    'Accuracy':  '#2196F3',
    'Precision': '#4CAF50',
    'Recall':    '#FF9800',
    'F1-Score':  '#E91E63',
}

METRICS

,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,88.3,87.1,89.6,88.3
1,Support Vector Machine,87.9,86.8,89.0,87.9
2,Random Forest,85.4,84.2,86.7,85.4
3,Naive Bayes,83.1,82.0,84.3,83.1
4,Decision Tree,78.2,77.5,79.0,78.2
5,KNN (k=5),73.6,72.1,75.2,73.6


## 3 · App Setup & Layout

Initialises the Dash app, defines the colour palette, and builds the top-level layout
with a header and two tabs.

In [4]:
app = Dash(__name__, title='Fake Review Detector',suppress_callback_exceptions=True)

# ── Colour palette ───────────────────────────────────────────────────────────
DARK   = '#0F1117'
CARD   = '#1A1D27'
ACCENT = '#6C63FF'
TEXT   = '#E0E0E0'
MUTED  = '#9E9E9E'


def card(children, style=None):
    """Reusable dark card container."""
    base = {
        'background': CARD,
        'borderRadius': '12px',
        'padding': '24px',
        'marginBottom': '20px',
        'border': '1px solid #2A2D3E',
    }
    if style:
        base.update(style)
    return html.Div(children, style=base)


app.layout = html.Div(
    style={'background': DARK, 'minHeight': '100vh',
           'fontFamily': "'Segoe UI', sans-serif", 'color': TEXT},
    children=[
        # ── Header ───────────────────────────────────────────────────────────
        html.Div(
            style={'background': f'linear-gradient(135deg, {ACCENT} 0%, #8B5CF6 100%)',
                   'padding': '32px 40px', 'marginBottom': '28px'},
            children=[
                html.H1('🔍 Fake Review Detection System',
                        style={'margin': 0, 'fontSize': '28px', 'fontWeight': '700', 'color': '#fff'}),
                html.P('NLP • Machine Learning • Sentiment Analysis',
                       style={'margin': '6px 0 0', 'color': 'rgba(255,255,255,0.75)', 'fontSize': '14px'}),
            ]
        ),

        html.Div(
            style={'maxWidth': '1200px', 'margin': '0 auto', 'padding': '0 24px'},
            children=[
                # ── Tabs ─────────────────────────────────────────────────────
                dcc.Tabs(
                    id='tabs', value='tab-analytics',
                    colors={'border': '#2A2D3E', 'primary': ACCENT, 'background': DARK},
                    style={'marginBottom': '24px'},
                    children=[
                        dcc.Tab(
                            label='📊  Model Analytics', value='tab-analytics',
                            style={'background': CARD, 'color': MUTED, 'border': 'none', 'padding': '12px 24px'},
                            selected_style={'background': DARK, 'color': TEXT,
                                            'border': f'2px solid {ACCENT}',
                                            'borderBottom': 'none', 'padding': '12px 24px'},
                        ),
                        dcc.Tab(
                            label='🔎  Review Analyzer', value='tab-analyzer',
                            style={'background': CARD, 'color': MUTED, 'border': 'none', 'padding': '12px 24px'},
                            selected_style={'background': DARK, 'color': TEXT,
                                            'border': f'2px solid {ACCENT}',
                                            'borderBottom': 'none', 'padding': '12px 24px'},
                        ),
                    ]
                ),
                html.Div(id='tab-content'),
            ]
        ),
    ]
)

## 4 · Tab Content Functions

Each function returns the HTML layout rendered inside the active tab.

In [5]:
def tab_analytics():
    """Model Analytics tab — KPI cards, bar chart, radar chart, metrics table."""
    return html.Div([
        # ── KPI row ──────────────────────────────────────────────────────────
        html.Div(
            style={'display': 'grid', 'gridTemplateColumns': 'repeat(4, 1fr)',
                   'gap': '16px', 'marginBottom': '24px'},
            children=[
                card([
                    html.Div(f"{METRICS.iloc[0]['Accuracy']}%",
                             style={'fontSize': '32px', 'fontWeight': '700',
                                    'color': METRIC_COLORS['Accuracy']}),
                    html.Div('Best Accuracy', style={'color': MUTED, 'fontSize': '13px'}),
                ]),
                card([
                    html.Div(f"{METRICS.iloc[0]['Precision']}%",
                             style={'fontSize': '32px', 'fontWeight': '700',
                                    'color': METRIC_COLORS['Precision']}),
                    html.Div('Best Precision', style={'color': MUTED, 'fontSize': '13px'}),
                ]),
                card([
                    html.Div(f"{METRICS.iloc[0]['Recall']}%",
                             style={'fontSize': '32px', 'fontWeight': '700',
                                    'color': METRIC_COLORS['Recall']}),
                    html.Div('Best Recall', style={'color': MUTED, 'fontSize': '13px'}),
                ]),
                card([
                    html.Div(f"{METRICS.iloc[0]['F1-Score']}%",
                             style={'fontSize': '32px', 'fontWeight': '700',
                                    'color': METRIC_COLORS['F1-Score']}),
                    html.Div('Best F1-Score', style={'color': MUTED, 'fontSize': '13px'}),
                ]),
            ]
        ),

        # ── Metric selector + bar chart ───────────────────────────────────────
        card([
            html.Div(
                style={'display': 'flex', 'justifyContent': 'space-between',
                       'alignItems': 'center', 'marginBottom': '16px'},
                children=[
                    html.H3('Model Comparison', style={'margin': 0}),
                    dcc.Dropdown(
                        id='metric-dropdown',
                        options=[{'label': m, 'value': m}
                                 for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']],
                        value='F1-Score',
                        clearable=False,
                        style={'width': '180px', 'background': DARK, 'color': '#000'},
                    ),
                ]
            ),
            dcc.Graph(id='metric-bar', config={'displayModeBar': False}),
        ]),

        # ── Radar chart ───────────────────────────────────────────────────────
        card([
            html.H3('All Metrics Radar', style={'marginBottom': '16px'}),
            dcc.Graph(id='radar-chart', figure=_radar_chart(),
                      config={'displayModeBar': False}),
        ]),

        # ── Metrics table ─────────────────────────────────────────────────────
        card([
            html.H3('Full Metrics Table', style={'marginBottom': '16px'}),
            dash_table.DataTable(
                data=METRICS.to_dict('records'),
                columns=[{'name': c, 'id': c} for c in METRICS.columns],
                style_table={'overflowX': 'auto'},
                style_header={'backgroundColor': '#2A2D3E', 'color': TEXT, 'fontWeight': 'bold'},
                style_cell={'backgroundColor': CARD, 'color': TEXT,
                            'border': '1px solid #2A2D3E',
                            'textAlign': 'center', 'padding': '10px'},
                style_data_conditional=[
                    {'if': {'row_index': 0}, 'backgroundColor': '#1E3A5F'},
                ],
                sort_action='native',
            ),
        ]),
    ])

In [6]:
def tab_analyzer():
    """Review Analyzer tab — model selector, text input, prediction output, examples."""
    return html.Div([
        card([
            html.H3('Classify a Review', style={'marginBottom': '8px'}),
            html.P(
                'Paste any product review below. '
                'The model will classify it as Genuine (OR) or Fake/Computer-Generated (CG).',
                style={'color': MUTED, 'fontSize': '14px', 'marginBottom': '16px'},
            ),

            html.Label('Select Model:', style={'fontSize': '13px', 'color': MUTED}),
            dcc.Dropdown(
                id='model-select',
                options=[{'label': n, 'value': n} for n in pipelines.keys()],
                value=list(pipelines.keys())[0],
                clearable=False,
                style={'marginBottom': '16px', 'color': '#000'},
            ),

            html.Label('Review Text:', style={'fontSize': '13px', 'color': MUTED}),
            dcc.Textarea(
                id='review-input',
                placeholder='e.g. "This product is absolutely amazing! Best purchase I have ever made."',
                style={
                    'width': '100%', 'height': '130px', 'background': '#0F1117',
                    'color': TEXT, 'border': '1px solid #2A2D3E', 'borderRadius': '8px',
                    'padding': '12px', 'fontSize': '14px', 'resize': 'vertical',
                    'boxSizing': 'border-box', 'marginBottom': '16px',
                },
            ),

            html.Button(
                '🔍  Analyze Review', id='analyze-btn', n_clicks=0,
                style={
                    'background': ACCENT, 'color': '#fff', 'border': 'none',
                    'borderRadius': '8px', 'padding': '12px 28px',
                    'fontSize': '15px', 'cursor': 'pointer', 'fontWeight': '600',
                },
            ),
        ]),

        html.Div(id='prediction-output'),

        # ── Quick test examples ───────────────────────────────────────────────
        card([
            html.H4('Quick Test Examples', style={'marginBottom': '12px'}),
            html.Div(
                style={'display': 'flex', 'gap': '12px', 'flexWrap': 'wrap'},
                children=[
                    html.Button(
                        label,
                        id={'type': 'example-btn', 'index': i},
                        n_clicks=0,
                        style={
                            'background': '#2A2D3E', 'color': TEXT,
                            'border': '1px solid #3A3D4E',
                            'borderRadius': '6px', 'padding': '8px 16px',
                            'cursor': 'pointer', 'fontSize': '13px',
                        }
                    )
                    for i, label in enumerate(['😍 Likely Fake', '😐 Neutral Review', '👎 Negative Review'])
                ]
            ),
        ]),
    ])

## 5 · Chart Helpers

Helper function that builds the **radar chart** comparing all models across every metric.

In [7]:
def _radar_chart():
    """Build a polar/radar chart for all models across the four metrics."""
    cats   = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors = ['#6C63FF', '#4CAF50', '#FF9800', '#E91E63', '#00BCD4', '#9C27B0']
    fig    = go.Figure()

    for i, row in METRICS.iterrows():
        vals = [row[c] for c in cats] + [row[cats[0]]]   # close the polygon
        fig.add_trace(go.Scatterpolar(
            r=vals, theta=cats + [cats[0]],
            name=row['Model'], mode='lines+markers',
            line=dict(color=colors[i % len(colors)], width=2),
            fill='toself', fillcolor=colors[i % len(colors)],
            opacity=0.15,
        ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, range=[60, 100], color=MUTED, gridcolor='#2A2D3E'),
            angularaxis=dict(color=TEXT, gridcolor='#2A2D3E'),
            bgcolor=CARD,
        ),
        paper_bgcolor=CARD, plot_bgcolor=CARD,
        font_color=TEXT,
        legend=dict(orientation='h', y=-0.15),
        margin=dict(t=20, b=60),
        height=380,
    )
    return fig

## 6 · Callbacks

Dash callbacks wire the interactive components together.
* **render_tab** — swaps the visible tab content.
* **update_bar** — re-draws the bar chart when a different metric is selected.
* **fill_example** — pre-fills the text area with a sample review.
* **analyze_review** — runs the selected model and renders the verdict card.

In [8]:
@app.callback(Output('tab-content', 'children'), Input('tabs', 'value'))
def render_tab(tab):
    """Swap tab content."""
    return tab_analytics() if tab == 'tab-analytics' else tab_analyzer()

In [9]:
@app.callback(
    Output('metric-bar', 'figure'),
    Input('metric-dropdown', 'value'),
)
def update_bar(metric):
    """Re-draw the horizontal bar chart for the chosen metric."""
    sorted_df = METRICS.sort_values(metric, ascending=True)
    color = METRIC_COLORS.get(metric, ACCENT)
    fig = go.Figure(go.Bar(
        x=sorted_df[metric], y=sorted_df['Model'],
        orientation='h', marker_color=color,
        text=[f'{v}%' for v in sorted_df[metric]],
        textposition='outside',
    ))
    fig.update_layout(
        xaxis=dict(range=[60, 100], color=MUTED, gridcolor='#2A2D3E'),
        yaxis=dict(color=TEXT),
        paper_bgcolor=CARD, plot_bgcolor=CARD,
        font_color=TEXT,
        margin=dict(l=10, r=60, t=10, b=10),
        height=300,
    )
    return fig

In [10]:
EXAMPLES = [
    'best product ever amazing love it perfect wonderful must buy highly recommend five stars',
    'the item arrived on time, packaging was decent, product quality is average for the price',
    'stopped working after one week, very poor build quality, customer support was unhelpful',
]


@app.callback(
    Output('review-input', 'value'),
    [Input({'type': 'example-btn', 'index': i}, 'n_clicks') for i in range(3)],
    prevent_initial_call=True,
)
def fill_example(*clicks):
    """Pre-fill the text area when an example button is clicked."""
    from dash import ctx
    if not ctx.triggered_id:
        return ''
    return EXAMPLES[ctx.triggered_id['index']]

In [11]:
@app.callback(
    Output('prediction-output', 'children'),
    Input('analyze-btn', 'n_clicks'),
    [State('review-input', 'value'), State('model-select', 'value')],
    prevent_initial_call=True,
)
def analyze_review(n_clicks, text, model_name):
    """Run the selected model on the entered text and render the verdict."""
    if not text or not text.strip():
        return card([html.P('⚠️ Please enter a review first.', style={'color': '#FF9800'})])

    pipe = pipelines.get(model_name)
    if pipe is None:
        return card([html.P(f'Model "{model_name}" not loaded.', style={'color': '#f44336'})])

    prediction = pipe.predict([text])[0]
    proba = (
        pipe.predict_proba([text])[0]
        if hasattr(pipe.named_steps['clf'], 'predict_proba')
        else None
    )

    is_fake       = prediction == 'CG'
    verdict_color = '#f44336' if is_fake else '#4CAF50'
    verdict_emoji = '🚩' if is_fake else '✅'
    verdict_label = 'FAKE / COMPUTER GENERATED' if is_fake else 'GENUINE / ORIGINAL'
    verdict_detail = (
        'This review shows patterns typical of generated or incentivised reviews.'
        if is_fake else
        'This review appears to be a genuine customer experience.'
    )

    children = [
        html.H3(f'{verdict_emoji} Verdict: {verdict_label}',
                style={'color': verdict_color, 'marginBottom': '8px'}),
        html.P(verdict_detail, style={'color': MUTED, 'marginBottom': '16px'}),
        html.P(f'Model used: {model_name}', style={'color': MUTED, 'fontSize': '13px'}),
    ]

    if proba is not None:
        classes    = pipe.classes_
        proba_dict = dict(zip(classes, proba))
        fig = go.Figure(go.Bar(
            x=list(proba_dict.keys()),
            y=[v * 100 for v in proba_dict.values()],
            marker_color=[
                verdict_color if k == prediction else '#2A2D3E'
                for k in proba_dict
            ],
            text=[f'{v*100:.1f}%' for v in proba_dict.values()],
            textposition='outside',
        ))
        fig.update_layout(
            yaxis=dict(range=[0, 110], title='Confidence (%)', color=MUTED, gridcolor='#2A2D3E'),
            xaxis=dict(ticktext=['CG (Fake)', 'OR (Genuine)'],
                       tickvals=list(proba_dict.keys()), color=TEXT),
            paper_bgcolor=CARD, plot_bgcolor=CARD,
            font_color=TEXT,
            margin=dict(t=10, b=10),
            height=250,
            title='Prediction Confidence',
        )
        children.append(dcc.Graph(figure=fig, config={'displayModeBar': False}))

    return card(children, style={'borderLeft': f'4px solid {verdict_color}'})

## 7 · Run the Dashboard

> ⚠️ `use_reloader=False` is **required** when running Dash inside Jupyter to prevent the
> kernel from restarting.
>
> After running the cell below, open **http://127.0.0.1:8050** in your browser.

In [12]:
if __name__ == '__main__':
    print('Dashboard running at: http://127.0.0.1:8050')
    app.run(debug=True, use_reloader=False)

Dashboard running at: http://127.0.0.1:8050
